# Reproducing Guo *et al.* 2026 (the theory paper of this package) — the imaginary-time mean-force state

**Paper**: C. Guo, W. Wu, X. Xu, T. Jiang, P.-X. Chen, R. Chen,
*Time-evolving matrix product operators for off-diagonal system-bath coupling*,
Phys. Rev. B **114**, 125413 (2026).

**Content reproduced**: the application of the PT-TEMPO/XTRG method on the imaginary-time contour in the paper — contracting the imaginary-time process tensor
yields the **mean-force state** of a strongly coupled quantum impurity:
$$\rho_{\mathrm{mfs}} = \frac{\mathrm{tr}_{b}\left[e^{-\beta H_{\mathrm{tot}}}\right]}{\mathrm{tr}\left[e^{-\beta H_{\mathrm{tot}}}\right]}$$
In the weak-coupling limit it reduces to the system Gibbs state $e^{-\beta H_s}/Z_s$; at finite coupling it deviates from the Gibbs state
(bath-induced corrections) — this is the central physics of strong-coupling thermodynamics (the mean-force Gibbs state).

**Model**: spin-1/2, $H_s = \Delta\sigma_z/2$, coupled through the conjugate pair $\sigma_+/2$ (**off-diagonal coupling**,
PT framework) to a sub-Ohmic bath
$J(\omega) = 2\pi\alpha\,\omega^{s}\omega_c^{1-s}$.

**Parameters**: $\Delta = 1$, $\beta = 1$ ($N=20$, $\delta\tau=0.05$), $s=0.5$, $\omega_c=5$,
$\chi=20$, with the coupling scanned over $\alpha \in \{0.05, 0.1, 0.2, 0.4\}$.

**Reference data**: the analytic weak-coupling Gibbs state $\rho_G = e^{-\beta H_s}/Z_s$.


In [ ]:
using TEMPO, ImpurityModelBase, LinearAlgebra

function mfs_population(; α, N=20, δτ=0.05, Δ=1.0, s=0.5, wc=5.0, chi=20)
    β = N * δτ
    trunc = truncdimcutoff(D=chi, ϵ=1.0e-10, add_back=0)
    lattice = PTLattice(N=N, δτ=δτ, contour=:imag)
    z = [-1 0; 0 1.0] / 2
    sp = [0 0; 1 0.0] / 2
    model = ImpurityHamiltonian(Δ .* z)
    mpsK = sysdynamics(lattice, model, trunc=trunc)
    hyb = NonDiagonalHyb(sp)                       # off-diagonal (conjugate-pair) coupling
    spec = spectrum(w -> 2π*α * w^s * wc^(1-s), lb=0, ub=wc)
    bath = bosonicbath(spec, β=β)
    corr = correlationfunction(bath, lattice)
    algmult = DMRGMult1(trunc)
    algexpan = OverDeterminedProny(n=20, tol=1.0e-8)
    alg = XTRGIF(k=5, fast=true, algmult=algmult, algexpan=algexpan)
    mpsI = hybriddynamics(lattice, corr, hyb, alg)
    mps = mult(mpsK, mpsI, trunc=trunc)
    ρ = meanforcestate(lattice, mps)
    ρ = ρ ./ tr(ρ)
    return real.(diag(ρ))    # [ground-state, excited-state] populations (H_s = Δσz/2, σz=-1 is ground)
end

# analytic weak-coupling limit
gibbs_population(; β=1.0, Δ=1.0) = begin
    z = [-1 0; 0 1.0] / 2
    ρG = exp(-β .* Δ .* z); ρG ./= tr(ρG)
    real.(diag(ρG))
end
println("functions defined")


In [ ]:
alphas = [0.05, 0.1, 0.2, 0.4]
mfs_pop = Dict{Float64, Vector{Float64}}()
for α in alphas
    @time mfs_pop[α] = mfs_population(α=α)
    println("α = ", α, "  populations = ", round.(mfs_pop[α], digits=4))
end
gibbs = gibbs_population()
println("weak-coupling Gibbs populations = ", round.(gibbs, digits=4))


In [ ]:
using Plots
pl = plot(xlabel="coupling α", ylabel="excited-state population",
          title="Mean-force state vs coupling (Guo et al. 2026, imaginary-time PT-TEMPO)",
          legend=:right, size=(640, 420))
exc = [mfs_pop[α][2] for α in alphas]
plot!(pl, alphas, exc, marker=:circle, lw=2, color=1, label="mean-force state (PT-TEMPO)")
hline!(pl, [gibbs[2]], color=2, ls=:dash, lw=2, label="weak-coupling Gibbs state (analytic)")
savefig(pl, "thermalstate_mfs.png")
pl


## Discussion of results

- Contracting the imaginary-time process tensor yields the thermal-equilibrium mean-force state $\rho_{\mathrm{mfs}}$;
  in the weak-coupling limit it approaches the Gibbs value (dashed line in the figure: excited-state population
  $\approx 0.269$ at $\beta\Delta = 1$). Even at the smallest $\alpha = 0.05$, the equilibrium correction induced by the sub-Ohmic bath ($s = 0.5$)
  is already visible ($0.34$) — a bath with large low-frequency spectral weight renormalizes the state appreciably at intermediate temperatures.
- As the coupling $\alpha$ increases, the excited-state population rises monotonically ($\approx 0.51$ at $\alpha = 0.4$) —
  this is the bath-induced strong-coupling equilibrium correction, i.e. the deviation of the **mean-force Gibbs state** from the standard Gibbs state,
  a key quantity in strong-coupling thermodynamics (quantum heat engines, modified Landauer principle, etc.).
- This reproduction follows the PT (process tensor) + translationally invariant influence functional (XTRG-style imaginary-time evolution) path,
  which is precisely the core algorithm of the paper for off-diagonal coupling.
